# LACRIMAE — F01 CANTOR
> *"Le Chantre transcrit. La mémoire de l'Ange devient données."*

**Mission** : Transcription `audio_clean.mp3` → `timing.json` via faster-whisper

**Prérequis** :
- Runtime Colab avec GPU T4 activé
- Drive monté avec `DRIVE_LACRIMAE/SHARED/audio_clean.mp3` présent
- `LAC_CUSTOS.py` présent dans `DRIVE_LACRIMAE/`

---

## ÉTAPE 1 — Montage Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive monté.')

## ÉTAPE 2 — Configuration des chemins

In [ ]:
from pathlib import Path

# ─── CHEMINS — adapter si besoin ────────────────────────────────────────────
DRIVE_BASE    = Path('/content/drive/MyDrive/DRIVE_LACRIMAE')
AUDIO_PATH    = DRIVE_BASE / 'SHARED' / 'audio_clean.mp3'
OUT_DIR       = DRIVE_BASE / 'F01_CANTOR' / 'OUT'
CODEBASE_DIR  = DRIVE_BASE / 'F01_CANTOR' / 'CODEBASE'
CUSTOS_PATH   = DRIVE_BASE / 'LAC_CUSTOS.py'

# ─── PARAMÈTRES ─────────────────────────────────────────────────────────────
MODEL_SIZE = 'medium'   # 'tiny' | 'medium' | 'large-v2'

OUT_DIR.mkdir(parents=True, exist_ok=True)

# Vérifications
assert AUDIO_PATH.exists(), f'Audio introuvable : {AUDIO_PATH}'
assert CUSTOS_PATH.exists(), f'LAC_CUSTOS introuvable : {CUSTOS_PATH}'

print(f'Audio    : {AUDIO_PATH}')
print(f'Output   : {OUT_DIR}')
print(f'Modèle   : {MODEL_SIZE}')
print('Chemins validés.')

## ÉTAPE 3 — Installation des dépendances

In [ ]:
!pip install faster-whisper -q
print('faster-whisper installé.')

## ÉTAPE 4 — Copie du script CANTOR

In [ ]:
import shutil
src = CODEBASE_DIR / 'lac_f01_cantor.py'
dst = Path('/content/lac_f01_cantor.py')
shutil.copy(src, dst)
print(f'Script copié → {dst}')

## ÉTAPE 5 — Transcription CANTOR

In [ ]:
import sys
sys.path.insert(0, '/content')

from lac_f01_cantor import transcribe, validate_timing, write_output

out_path = str(OUT_DIR / 'timing.json')

timing = transcribe(str(AUDIO_PATH), model_size=MODEL_SIZE)

if validate_timing(timing):
    write_output(timing, out_path)
    print(f'\n[CANTOR] timing.json produit : {out_path}')
    print(f'[CANTOR] Durée  : {timing["audio_duration_s"]}s')
    print(f'[CANTOR] Frames : {timing["total_frames"]}')
    print(f'[CANTOR] Mots   : {len(timing["words"])}')
else:
    print('[CANTOR] ÉCHEC — vérifier les logs ci-dessus.')

## ÉTAPE 6 — Aperçu des 10 premiers mots

In [ ]:
import json
with open(out_path, 'r', encoding='utf-8') as f:
    t = json.load(f)

print(f'Durée audio : {t["audio_duration_s"]}s | Total frames : {t["total_frames"]} | FPS : {t["fps"]}')
print(f'Nombre de mots : {len(t["words"])}\n')
print(f'{"MOT":<20} {"START_S":>8} {"END_S":>7} {"F_START":>7} {"F_END":>7} {"FORT":>5}')
print('-' * 60)
for w in t['words'][:10]:
    print(f'{w["word"]:<20} {w["start_s"]:>8.3f} {w["end_s"]:>7.3f} {w["start_frame"]:>7} {w["end_frame"]:>7} {str(w["is_strong"]):>5}')

## ÉTAPE 7 — Validation LAC_CUSTOS (check-out F01)

In [ ]:
import shutil
shutil.copy(CUSTOS_PATH, '/content/LAC_CUSTOS.py')

!python /content/LAC_CUSTOS.py --frigate F01 --mode check-out --drive-base "{DRIVE_BASE}"

## ÉTAPE 8 — Instructions de transit

```
✓ Si LAC_CUSTOS a validé :

  Copier manuellement :
  F01_CANTOR/OUT/timing.json  →  F02_VISIO/IN/timing.json
  F01_CANTOR/OUT/timing.json  →  F03_PICTOR/IN/timing.json
  F01_CANTOR/OUT/timing.json  →  F04_SIGNUM/IN/timing.json

  Puis inscrire le transit dans TRACKING/LACRIMAE_TRANSFER_LOG.md
```